# H2 triplet M_s-invariance checks

This notebook uses the H2 geometry and active space from h2_mce.ipynb. It evaluates all three triplet components (M_s = -1, 0, +1) and verifies that Forte2 spin-free RDMs, the requested two-cumulant, and the MCA two-fragment energy are invariant.

In [1]:
import numpy as np
import forte2
import mutual_correlation_energy_mod as mca_reference
import mutual_correlation_energy_mod_fast as mca_fast

ATOL = 1.0e-10
RTOL = 1.0e-10

In [2]:
xyz = """
H 0.000 0.000 0.000
H 0.000 0.000 1.740
"""

system = forte2.System(
    xyz=xyz, basis_set="cc-pVDZ", auxiliary_basis_set="cc-pVTZ-JKFIT"
)
rhf = forte2.RHF(charge=0)(system)
rhf.run()

def make_triplet_ci(ms):
    ci = forte2.CI(
        forte2.State(system=system, multiplicity=3, ms=ms),
        active_orbitals=[0, 1],
    )(rhf)
    ci.run()
    return ci

triplet = {ms: make_triplet_ci(ms) for ms in (-1.0, 0.0, 1.0)}

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   H   0.00000000   0.00000000   0.00000000
   H   0.00000000   0.00000000   3.28812346
Parsed 2 atoms with basis set of 10 functions.
  Max eigenvalue: 2.268e+00
  Min eigenvalue: 1.969e-01
  Condition number: 1.152e+01
  Inverse condition number: 8.683e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 10
  Largest discarded eigenvalue: 0.000e+00


  Smallest kept eigenvalue: 1.969e-01
Number of electrons: 2
Number of alpha electrons: 1
Number of beta electrons: 1
Ms: 0
Total charge: 0
Number of basis functions: 10
Number of orthogonalized basis functions: 10
Number of auxiliary basis functions: 60
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis functions: 10
Number of auxiliary basis functions: 60
Iter               Energy           ΔE       ||ΔD||  ||AO grad||      <S^2>  DIIS
---------------------------------------------------------------------------------
   1      -0.960429111551  -7.0124e-04   2.6631e-02   2.5316e-02    0.00000     S
   2      -0.960455422838  -2.6311e-05   6.2306e-03   4.8306e-03    0.00000   S/E
   3      -0.960455426123  -3.2852e-09   3.0424e-05   6.8500e-05    0.00000   S/E
   4      -0.960455426123  -1.3323e-15   4.6473e-08   3.6729e-

In [3]:
def spin_free_cumulant(ci):
    solver = ci.sub_solvers[0]
    gamma1 = solver.make_sf_1rdm(0)
    gamma2 = solver.make_sf_2rdm(0)
    cumulant = (
        gamma2
        - np.einsum("pr,qs->pqrs", gamma1, gamma1)
        + 0.5 * np.einsum("ps,qr->pqrs", gamma1, gamma1)
    )
    return gamma1, gamma2, cumulant

rdms = {ms: spin_free_cumulant(ci) for ms, ci in triplet.items()}
reference_ms = 0.0
gamma1_ref, gamma2_ref, lambda_ref = rdms[reference_ms]

In [4]:
print("Maximum absolute deviations from M_s = 0")
for ms, (gamma1, gamma2, cumulant) in rdms.items():
    print(
        f"M_s = {ms:+.0f}: gamma1 {np.max(np.abs(gamma1 - gamma1_ref)):.3e}, "
        f"gamma2 {np.max(np.abs(gamma2 - gamma2_ref)):.3e}, "
        f"lambda {np.max(np.abs(cumulant - lambda_ref)):.3e}"
    )
    np.testing.assert_allclose(gamma1, gamma1_ref, rtol=RTOL, atol=ATOL)
    np.testing.assert_allclose(gamma2, gamma2_ref, rtol=RTOL, atol=ATOL)
    np.testing.assert_allclose(cumulant, lambda_ref, rtol=RTOL, atol=ATOL)

print("Spin-free RDM and cumulant M_s-invariance: PASS")

Maximum absolute deviations from M_s = 0
M_s = -1: gamma1 2.220e-16, gamma2 2.220e-16, lambda 2.220e-16
M_s = +0: gamma1 0.000e+00, gamma2 0.000e+00, lambda 0.000e+00
M_s = +1: gamma1 2.220e-16, gamma2 2.220e-16, lambda 2.220e-16
Spin-free RDM and cumulant M_s-invariance: PASS


In [5]:
# Historical implementation: reconstruct the spin-summed cumulant from
# spin-dependent RDM blocks, as the MCA modules did before this update.
def old_spin_dependent_cumulant(ci):
    solver = ci.sub_solvers[0]
    alpha1, beta1 = solver.make_sd_1rdm(0)
    aa_pair, ab2, bb_pair = solver.make_sd_2rdm(0)
    aa2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(aa_pair)
    bb2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(bb_pair)

    aa_cumulant = (
        aa2
        - np.einsum("pr,qs->pqrs", alpha1, alpha1)
        + np.einsum("ps,qr->pqrs", alpha1, alpha1)
    )
    ab_cumulant = ab2 - np.einsum("pr,qs->pqrs", alpha1, beta1)
    bb_cumulant = (
        bb2
        - np.einsum("pr,qs->pqrs", beta1, beta1)
        + np.einsum("ps,qr->pqrs", beta1, beta1)
    )
    return (
        aa_cumulant
        + ab_cumulant
        + ab_cumulant.transpose(1, 0, 3, 2)
        + bb_cumulant
    )

old_lambdas = {ms: old_spin_dependent_cumulant(ci) for ms, ci in triplet.items()}
old_reference = old_lambdas[reference_ms]

print("Old spin-dependent reconstruction versus M_s = 0")
for ms, old_lambda in old_lambdas.items():
    ms_delta = np.max(np.abs(old_lambda - old_reference))
    formula_delta = np.max(np.abs(old_lambda - rdms[ms][2]))
    print(
        f"M_s = {ms:+.0f}: old-formula M_s delta {ms_delta:.3e}, "
        f"old-vs-spin-free delta {formula_delta:.3e}"
    )

# This confirms the old construction is M_s dependent for the triplet.
assert np.max(np.abs(old_lambdas[-1.0] - old_reference)) > ATOL
assert np.max(np.abs(old_lambdas[1.0] - old_reference)) > ATOL
#print("Old spin-dependent reconstruction is M_s dependent: CONFIRMED")


Old spin-dependent reconstruction versus M_s = 0
M_s = -1: old-formula M_s delta 5.000e-01, old-vs-spin-free delta 5.000e-01
M_s = +0: old-formula M_s delta 0.000e+00, old-vs-spin-free delta 0.000e+00
M_s = +1: old-formula M_s delta 5.000e-01, old-vs-spin-free delta 5.000e-01


In [6]:
# Both MCA modules must implement the same spin-free formula.
for ms, ci in triplet.items():
    expected = rdms[ms][2]
    np.testing.assert_allclose(
        mca_reference._spin_summed_2cumulant(ci)[2], expected, rtol=RTOL, atol=ATOL
    )
    mca_fast.clear_cache()
    np.testing.assert_allclose(
        mca_fast._spin_summed_2cumulant(ci), expected, rtol=RTOL, atol=ATOL
    )
    np.testing.assert_allclose(
        mca_fast._spin_summed_2cumulant(ci), expected, rtol=RTOL, atol=ATOL
    )

print("Reference and cached MCA cumulant builders: PASS")

Reference and cached MCA cumulant builders: PASS


In [7]:
def two_fragment_energy(module, ci):
    return module.twofrag_correlation_energy_enumerated(ci, [0], [1], root=0)

m2_reference = {}
m2_fast = {}
for ms, ci in triplet.items():
    m2_reference[ms] = two_fragment_energy(mca_reference, ci)
    mca_fast.clear_cache()
    m2_fast[ms] = two_fragment_energy(mca_fast, ci)
    np.testing.assert_allclose(m2_fast[ms], m2_reference[ms], rtol=RTOL, atol=ATOL)

for ms in (-1.0, 0.0, 1.0):
    np.testing.assert_allclose(m2_reference[ms], m2_reference[reference_ms], rtol=RTOL, atol=ATOL)
    print(f"M_s = {ms:+.0f}: M2 = {m2_reference[ms]:+.12f} Eh")

print("MCA two-fragment M_s-invariance and fast/reference agreement: PASS")

M_s = -1: M2 = -0.069845686760 Eh
M_s = +0: M2 = -0.069845686760 Eh
M_s = +1: M2 = -0.069845686760 Eh
MCA two-fragment M_s-invariance and fast/reference agreement: PASS


All assertions use a tight 1e-10 absolute and relative tolerance. If a future Forte2 change alters the RDM convention, this notebook identifies whether the discrepancy is in the RDMs, cumulant construction, or MCA contraction.